# Dictionaries and nested data

## Learning objectives

By the end of this notebook you will be able to:

- create dictionaries and look up values by key
- add, change and remove entries safely
- loop over keys, values and items
- count and group with dictionaries
- work with nested structures: lists in dictionaries and dictionaries in lists

## Introduction

A dictionary maps **keys** to **values**: a patient ID to a record, a code to a label, a test name to a
result. Looking up by key is fast and readable. Nested dictionaries and lists are also the shape of most
data that comes from web services and JSON files.

## Creating and reading a dictionary

In [1]:
patient = {
    "id": "P0042",
    "age": 34,
    "sex": "F",
    "site": "Site B",
}
print(patient["age"])
print(patient)

34
{'id': 'P0042', 'age': 34, 'sex': 'F', 'site': 'Site B'}


A missing key raises `KeyError`. `get` returns `None` (or a default you choose) instead:

In [2]:
try:
    patient["weight"]
except KeyError as e:
    print("KeyError:", e)
print(patient.get("weight"), patient.get("weight", "not recorded"))
print("age" in patient, "weight" in patient)

KeyError: 'weight'
None not recorded
True False


Keys must be immutable (strings, numbers, tuples). Values can be anything, including lists and other
dictionaries. Since Python 3.7 dictionaries keep the order in which keys were added.

## Adding, changing and removing

In [3]:
patient["weight_kg"] = 61.5          # add
patient["age"] = 35                  # change
patient.update({"smoker": False, "site": "Site A"})
print(patient)

removed = patient.pop("smoker")      # remove and return the value
del patient["site"]
print(removed, patient)

{'id': 'P0042', 'age': 35, 'sex': 'F', 'site': 'Site A', 'weight_kg': 61.5, 'smoker': False}
False {'id': 'P0042', 'age': 35, 'sex': 'F', 'weight_kg': 61.5}


## Looping

In [4]:
for key in patient:
    print(key)

for key, value in patient.items():
    print(f"{key:>10}: {value}")

print(list(patient.keys()), list(patient.values()))

id
age
sex
weight_kg
        id: P0042
       age: 35
       sex: F
 weight_kg: 61.5
['id', 'age', 'sex', 'weight_kg'] ['P0042', 35, 'F', 61.5]


## Dictionaries as look-up tables

Translating codes into labels, or messy spellings into clean ones:

In [5]:
labels = {"P": "Positive", "N": "Negative", "I": "Inconclusive"}
codes = ["P", "N", "N", "I", "X", "P"]
print([labels.get(c, "Unknown") for c in codes])

unit_factor = {"mmol/L": 1.0, "mg/dL": 1 / 18}
readings = [(5.6, "mmol/L"), (126, "mg/dL"), (7.1, "mmol/L")]
print([round(v * unit_factor[u], 1) for v, u in readings])

['Positive', 'Negative', 'Negative', 'Inconclusive', 'Unknown', 'Positive']
[5.6, 7.0, 7.1]


## Counting

Counting how often each value occurs is one of the most common tasks.

In [6]:
results = ["Neg", "Pos", "Neg", "Neg", "Pos", "Inc", "Neg"]
counts = {}
for r in results:
    counts[r] = counts.get(r, 0) + 1
print(counts)

from collections import Counter
c = Counter(results)
print(c, c.most_common(1))

{'Neg': 4, 'Pos': 2, 'Inc': 1}
Counter({'Neg': 4, 'Pos': 2, 'Inc': 1}) [('Neg', 4)]


## Grouping

Collecting values into lists by a key:

In [7]:
samples = [("Site A", 5.4), ("Site B", 7.8), ("Site A", 6.1), ("Site C", 9.2), ("Site B", 5.0)]

by_site = {}
for site, value in samples:
    by_site.setdefault(site, []).append(value)
print(by_site)
print({site: round(sum(v) / len(v), 2) for site, v in by_site.items()})

{'Site A': [5.4, 6.1], 'Site B': [7.8, 5.0], 'Site C': [9.2]}
{'Site A': 5.75, 'Site B': 6.4, 'Site C': 9.2}


`setdefault(key, [])` returns the list for that key, creating an empty one the first time.
`collections.defaultdict(list)` does the same automatically.

## Dictionary comprehensions

In [8]:
squares = {n: n ** 2 for n in range(1, 6)}
high_only = {pid: g for pid, g in {"P1": 5.4, "P2": 7.8, "P3": 9.2}.items() if g >= 7}
print(squares, high_only)

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25} {'P2': 7.8, 'P3': 9.2}


## Sorting a dictionary by its values

In [9]:
positivity = {"Site A": 0.12, "Site B": 0.31, "Site C": 0.08}
for site in sorted(positivity, key=positivity.get, reverse=True):
    print(site, positivity[site])

Site B 0.31
Site A 0.12
Site C 0.08


## Nested data structures

### A dictionary containing lists and dictionaries

In [10]:
record = {
    "id": "P0042",
    "demographics": {"age": 34, "sex": "F"},
    "visits": [
        {"date": "2025-01-10", "glucose": 7.8, "flags": ["fasting"]},
        {"date": "2025-02-09", "glucose": 6.9, "flags": []},
        {"date": "2025-03-11", "glucose": 6.2, "flags": ["repeat"]},
    ],
}
print(record["demographics"]["age"])
print(record["visits"][0]["glucose"])
print([v["glucose"] for v in record["visits"]])
print(record["visits"][-1]["flags"][0])

34
7.8
[7.8, 6.9, 6.2]
repeat


Read a path like `record["visits"][0]["glucose"]` from left to right: the record, its visits list,
the first visit, its glucose value.

### A list of dictionaries: one per record

This is the most common shape for data before it becomes a table.

In [11]:
participants = [
    {"id": "P01", "age": 34, "site": "Site A", "glucose": 5.4},
    {"id": "P02", "age": 51, "site": "Site B", "glucose": 7.8},
    {"id": "P03", "age": 47, "site": "Site A", "glucose": 6.1},
    {"id": "P04", "age": 62, "site": "Site C"},                 # glucose missing
]
ages = [p["age"] for p in participants]
high = [p["id"] for p in participants if p.get("glucose", 0) >= 7]
print(ages, high)

site_a = [p for p in participants if p["site"] == "Site A"]
print(len(site_a), "participants at Site A")

[34, 51, 47, 62] ['P02']
2 participants at Site A


### Extracting safely from nested data

Real nested data often has missing pieces. A small helper avoids long chains of `if` checks:

In [12]:
def get_path(data, path, default=None):
    for step in path:
        try:
            data = data[step]
        except (KeyError, IndexError, TypeError):
            return default
    return data

print(get_path(record, ["visits", 2, "flags", 0]))
print(get_path(record, ["visits", 5, "glucose"], "no such visit"))
print(get_path(record, ["demographics", "weight"], "not recorded"))

repeat
no such visit
not recorded


### From nested data to a table

pandas turns a list of dictionaries straight into a table (module 6):

In [13]:
import pandas as pd
pd.DataFrame(participants)

,id,age,site,glucose
0,P01,34,Site A,5.4
1,P02,51,Site B,7.8
2,P03,47,Site A,6.1
3,P04,62,Site C,NaN


## JSON

JSON, the standard format for web data, maps directly onto dictionaries and lists.

In [14]:
import json
text = json.dumps(record, indent=2)
print(text[:200])
back = json.loads(text)
print(back["visits"][1]["glucose"])

{
  "id": "P0042",
  "demographics": {
    "age": 34,
    "sex": "F"
  },
  "visits": [
    {
      "date": "2025-01-10",
      "glucose": 7.8,
      "flags": [
        "fasting"
      ]
    },
    {

6.9


## Practise

Exercises for this folder are in `exercises/Data_Structures_Exercises.ipynb`.

## Further reading

- [Python tutorial: dictionaries](https://docs.python.org/3/tutorial/datastructures.html#dictionaries)
- [collections: Counter, defaultdict](https://docs.python.org/3/library/collections.html)
- [json module](https://docs.python.org/3/library/json.html)